# Multi-label Emotion Classification with BERT

This notebook fine-tunes `bert-base-uncased` on the Hugging Face GoEmotions training split. It preserves examples with at least one non-neutral emotion and models the resulting **27 non-neutral emotion labels**. Neutral-only examples are excluded because this project keeps the original non-neutral objective.

The final test partition is not used for checkpoint selection or threshold decisions. Full execution downloads the dataset and pretrained model and may require a GPU.

## 1. Reproducible setup


In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from transformers import (
    BertForSequenceClassification,
    BertTokenizerFast,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128
THRESHOLD = 0.5
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"PyTorch device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


W1026 17:13:29.031000 18768 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


PyTorch device: cuda


## 2. Load and validate GoEmotions


In [2]:
dataset = load_dataset("go_emotions", "raw", split="train[:7000]")
df = pd.DataFrame(dataset)

GO_EMOTIONS_LABELS = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval", "disgust",
    "embarrassment", "excitement", "fear", "gratitude", "grief", "joy", "love",
    "nervousness", "optimism", "pride", "realization", "relief", "remorse",
    "sadness", "surprise", "neutral",
]
missing_labels = set(GO_EMOTIONS_LABELS) - set(df.columns)
if missing_labels:
    raise ValueError(f"GoEmotions schema is missing labels: {sorted(missing_labels)}")
emotion_labels = [label for label in GO_EMOTIONS_LABELS if label != "neutral"]
df = df[df["text"].notna() & df["text"].str.strip().ne("")].copy()
df = df[df[emotion_labels].sum(axis=1) > 0].reset_index(drop=True)
print(f"Loaded rows: {len(dataset)}")
print(f"Usable non-neutral rows: {len(df)}")
print(f"Labels: {len(emotion_labels)}")
print(f"Duplicate texts: {df['text'].duplicated().sum()}")


Using the latest cached version of the dataset since go_emotions couldn't be found on the Hugging Face Hub


Found the latest cached dataset configuration 'raw' at C:\Users\moham\.cache\huggingface\datasets\go_emotions\raw\0.0.0\add492243ff905527e67aeb8b80c082af02207c3 (last modified on Wed May 14 10:57:08 2025).


Loaded rows: 7000
Usable non-neutral rows: 5043
Labels: 27
Duplicate texts: 204


## 3. Grouped train/validation/test split

The split is performed on unique text values rather than rows. This prevents identical text from appearing in more than one partition. The proportions are approximately 80% training, 10% validation, and 10% final test. Multi-label stratification is not applied; label distributions are printed for inspection.

In [3]:
unique_texts = pd.Series(df["text"].unique(), name="text")
train_texts, heldout_texts = train_test_split(
    unique_texts, test_size=0.2, random_state=SEED
)
validation_texts, test_texts = train_test_split(
    heldout_texts, test_size=0.5, random_state=SEED
)
train_text_set = set(train_texts)
validation_text_set = set(validation_texts)
test_text_set = set(test_texts)
if train_text_set & (validation_text_set | test_text_set) or validation_text_set & test_text_set:
    raise AssertionError("Text overlap detected between partitions")

train_df = df[df["text"].isin(train_text_set)].copy()
validation_df = df[df["text"].isin(validation_text_set)].copy()
test_df = df[df["text"].isin(test_text_set)].copy()
print(f"Train: {len(train_df)} ({len(train_df) / len(df):.1%})")
print(f"Validation: {len(validation_df)} ({len(validation_df) / len(df):.1%})")
print(f"Test: {len(test_df)} ({len(test_df) / len(df):.1%})")
for name, part in [("train", train_df), ("validation", validation_df), ("test", test_df)]:
    print(name, part[emotion_labels].sum().sort_values(ascending=False).head(5).to_dict())


Train: 4041 (80.1%)
Validation: 500 (9.9%)
Test: 502 (10.0%)
train {'approval': 471, 'admiration': 428, 'annoyance': 357, 'gratitude': 324, 'disapproval': 307}
validation {'approval': 77, 'annoyance': 43, 'admiration': 42, 'amusement': 36, 'curiosity': 34}
test {'approval': 58, 'admiration': 54, 'gratitude': 43, 'curiosity': 42, 'disapproval': 40}


## 4. Tokenization and PyTorch datasets


In [4]:
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_frame(frame):
    return tokenizer(
        frame["text"].tolist(), truncation=True, padding=True,
        max_length=MAX_LENGTH, return_tensors="pt"
    )

class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.to_numpy(dtype=np.float32)

    def __getitem__(self, index):
        item = {key: value[index] for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[index], dtype=torch.float32)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = EmotionDataset(tokenize_frame(train_df), train_df[emotion_labels])
validation_dataset = EmotionDataset(tokenize_frame(validation_df), validation_df[emotion_labels])
test_dataset = EmotionDataset(tokenize_frame(test_df), test_df[emotion_labels])


## 5. BERT model and validation-based checkpoint selection


In [5]:
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(emotion_labels),
    problem_type="multi_label_classification",
)

def multilabel_metrics(prediction):
    probabilities = torch.sigmoid(torch.tensor(prediction.predictions)).numpy()
    predictions = (probabilities >= THRESHOLD).astype(int)
    labels = prediction.label_ids.astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="micro", zero_division=0
    )
    _, _, macro_f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    return {"f1_micro": f1, "precision_micro": precision, "recall_micro": recall, "f1_macro": macro_f1}

training_args = TrainingArguments(
    output_dir="results",
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    fp16=torch.cuda.is_available(),
)
trainer = Trainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=validation_dataset, compute_metrics=multilabel_metrics,
)
trainer.train()
trainer.save_model("emotion_model")
tokenizer.save_pretrained("emotion_model")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Micro,Precision Micro,Recall Micro,F1 Macro
1,0.193488,0.164633,0.064801,0.880000,0.033639,0.029101
2,0.149462,0.149743,0.126227,0.762712,0.068807,0.064053
3,0.128324,0.146523,0.227560,0.656934,0.137615,0.111597
4,0.107077,0.143769,0.301050,0.635468,0.197248,0.168717
5,0.090395,0.143952,0.337719,0.596899,0.235474,0.211879
6,0.078381,0.145837,0.350691,0.574913,0.252294,0.230222


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

[transformers] There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.encoder.layer.0.output.LayerNorm.beta', 'bert.encoder.layer.0.output.LayerNorm.gamma', 'bert.encoder.layer.1.attention.output.LayerNorm.beta', 'bert.encoder.layer.1.attention.output.LayerNorm.gamma', 'bert.encoder.layer.1.output.LayerNorm.beta', 'bert.encoder.layer.1.output.LayerNorm.gamma', 'bert.encoder.layer.2.attention.output.LayerNorm.beta', 'bert.encoder.layer.2.attention.output.LayerNorm.gamma', 'bert.encoder.layer.2.output.LayerNorm.beta', 'bert.encoder.layer.2.output.LayerNorm.gamma', 'bert.encoder.layer.3.attention.output.LayerNorm.beta', 'bert.encoder.layer.3.attention.output.LayerNorm.gamma', 'bert.encoder.layer.3.output.LayerNorm.beta', 'bert.encoder.layer.3.output.LayerNorm.gamma', 'bert.encoder.layer.4.attention.

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('emotion_model\\tokenizer_config.json', 'emotion_model\\tokenizer.json')

## 6. Untouched final test evaluation

Run this section only after training. The test set has not been passed to `Trainer` and is not used for checkpoint selection or threshold tuning.

In [6]:
def evaluate_test_set(trained_trainer):
    predictions = trained_trainer.predict(test_dataset)
    probabilities = torch.sigmoid(torch.tensor(predictions.predictions)).numpy()
    predicted_labels = (probabilities >= THRESHOLD).astype(int)
    true_labels = predictions.label_ids.astype(int)
    for average in ("micro", "macro", "weighted"):
        precision, recall, f1, _ = precision_recall_fscore_support(
            true_labels, predicted_labels, average=average, zero_division=0
        )
        print(f"{average}: precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")
    print("Labels with no positive test examples:", int((true_labels.sum(axis=0) == 0).sum()))
    print(classification_report(
        true_labels, predicted_labels, target_names=emotion_labels,
        zero_division=0, output_dict=False,
    ))

evaluate_test_set(trainer)


micro: precision=0.5960, recall=0.2826, f1=0.3834
macro: precision=0.3844, recall=0.1904, f1=0.2368
weighted: precision=0.5152, recall=0.2826, f1=0.3432
Labels with no positive test examples: 0
                precision    recall  f1-score   support

    admiration       0.60      0.46      0.52        54
     amusement       0.81      0.57      0.67        30
         anger       1.00      0.19      0.32        16
     annoyance       0.31      0.14      0.19        36
      approval       0.47      0.33      0.39        58
        caring       0.33      0.12      0.18        16
     confusion       0.67      0.11      0.20        35
     curiosity       0.56      0.36      0.43        42
        desire       0.00      0.00      0.00        11
disappointment       0.33      0.07      0.11        15
   disapproval       0.36      0.23      0.28        40
       disgust       0.00      0.00      0.00        15
 embarrassment       0.00      0.00      0.00        10
    excitement       

## 7. Inference and limitations

The optional `app.py` loads a saved `emotion_model` directory and does not retrain. The final metrics are intentionally not embedded in this notebook because full corrected training has not been run yet.